In [1]:
# 环境与路径配置
from pathlib import Path
import tarfile
import json
from typing import List, Tuple

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

PROJECT_ROOT = Path(r"/root/autodl-tmp/Sharebottom_PEPNet")
RAW_DIR = PROJECT_ROOT / "Ali-CCP"
WORK_DIR = PROJECT_ROOT / "datapreprocess" / "workspace"
EXTRACT_DIR = WORK_DIR / "raw_extracted"
OUT_DIR = WORK_DIR / "processed"
REPORT_DIR = WORK_DIR / "reports"

for d in [WORK_DIR, EXTRACT_DIR, OUT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("WORK_DIR:", WORK_DIR)

PROJECT_ROOT: /root/autodl-tmp/Sharebottom_PEPNet
RAW_DIR: /root/autodl-tmp/Sharebottom_PEPNet/Ali-CCP
WORK_DIR: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace


In [ ]:
# # 解压 tar.gz 文件
# archives = sorted(RAW_DIR.glob("*.tar.gz"))
# print("Found archives:")
# for a in archives:
#     print(" -", a.name)

# for arc in archives:
#     target = EXTRACT_DIR / arc.stem.replace('.tar', '')
#     target.mkdir(parents=True, exist_ok=True)
#     with tarfile.open(arc, "r:gz") as tf:
#         tf.extractall(target)
#     print(f"Extracted: {arc.name} -> {target}")

# print("\nExtract done.")

In [2]:
# 扫描解压目录并推断文件角色（skeleton/common）
all_files = [p for p in EXTRACT_DIR.rglob('*') if p.is_file()]
print(f"Total extracted files: {len(all_files)}")
for p in all_files[:30]:
    print(" -", p)

def guess_role(path: Path) -> str:
    name = path.name.lower()
    if ('skeleton' in name) or (('sample' in name) and ('common' not in name)):
        return 'skeleton'
    if 'common' in name:
        return 'common'
    return 'unknown'

role_map = [(p, guess_role(p)) for p in all_files]
role_df = pd.DataFrame([(str(p), r) for p, r in role_map], columns=['path', 'role'])

display(role_df.groupby('role').size().rename('count').reset_index())
display(role_df.head(50))

Total extracted files: 4
 - /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_test/common_features_test.csv
 - /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_test/sample_skeleton_test.csv
 - /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_train/common_features_train.csv
 - /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_train/sample_skeleton_train.csv


,role,count
0,common,2
1,skeleton,2


,path,role
0,/root/autodl-tmp/Sharebottom_PEPNet/dataprepro...,common
1,/root/autodl-tmp/Sharebottom_PEPNet/dataprepro...,skeleton
2,/root/autodl-tmp/Sharebottom_PEPNet/dataprepro...,common
3,/root/autodl-tmp/Sharebottom_PEPNet/dataprepro...,skeleton


In [3]:
# 手工指定 train/test 的 skeleton/common 文件路径（明确指定解压后的4个CSV文件）
TRAIN_SKELETON = EXTRACT_DIR / "sample_train" / "sample_skeleton_train.csv"
TRAIN_COMMON = EXTRACT_DIR / "sample_train" / "common_features_train.csv"
TEST_SKELETON = EXTRACT_DIR / "sample_test" / "sample_skeleton_test.csv"
TEST_COMMON = EXTRACT_DIR / "sample_test" / "common_features_test.csv"

# 此处打印检查，避免错误
print("TRAIN_SKELETON:", TRAIN_SKELETON, "Exists:", TRAIN_SKELETON.exists())
print("TRAIN_COMMON:", TRAIN_COMMON, "Exists:", TRAIN_COMMON.exists())
print("TEST_SKELETON:", TEST_SKELETON, "Exists:", TEST_SKELETON.exists())
print("TEST_COMMON:", TEST_COMMON, "Exists:", TEST_COMMON.exists())

assert TRAIN_SKELETON.exists(), "找不到 TRAIN_SKELETON"
assert TRAIN_COMMON.exists(), "找不到 TRAIN_COMMON"
assert TEST_SKELETON.exists(), "找不到 TEST_SKELETON"
assert TEST_COMMON.exists(), "找不到 TEST_COMMON"

TRAIN_SKELETON: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_train/sample_skeleton_train.csv Exists: True
TRAIN_COMMON: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_train/common_features_train.csv Exists: True
TEST_SKELETON: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_test/sample_skeleton_test.csv Exists: True
TEST_COMMON: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/raw_extracted/sample_test/common_features_test.csv Exists: True


### 模型输入处理 (对齐 Sharebottom + PEPNet 架构)

接下来的处理紧扣您的架构设计：
1. **统一解析并构建全局词表 (Vocab Generator)**：过滤低频词，统一将 `(field_id, feature_id)` 映射为从 2 开始的稠密整数索引，0给 Padding，1给 OOV (Out of Vocabulary)。
2. **特征分流映射 (Feature Routing)**：在单条样本遍历合并 `skeleton` 与 `common` 时，依据定义的四个大类分组，把特征提取并分块写入最终格式里，使得 `DataLoader` 只需原样吐出张量：
    - `'epnet_scene'` $\rightarrow$ **EPNet 场景特异性底座**
    - `'user_profile'` / `'user_behavior'` $\rightarrow$ **Gate NN 个性化滤镜 & PPNet 参数微调**
    - `'item_and_cross'` $\rightarrow$ **大哥塔 (Click Tower) 底层共享底座**
3. **清洗与导出**：过滤掉所有 `y=0, z=1` 的非法标签，并且额外生成 `sample_weight_conv` 对抗严重的不平衡问题，最后将每个样本导成 Parquet。

In [ ]:
import gc
import pickle
import json
from collections import defaultdict
import pyarrow as pa
import pyarrow.parquet as pq
import sqlite3

# ========================================
# 1. 核心解析逻辑与架构强相关的特征划分
# ========================================
SEP_LIST = '\x01' # 快递列表分隔符
SEP_FID = '\x02'  # 区分“这是什么类型的包裹(Field)”和“里面装了什么(ID)”的分隔符
SEP_IDV = '\x03'  # 区分“具体是什么ID”和“它的重要程度是多少(Value)”的分隔符

#将特定 field 强制定向到 4 个网络组件路径
FIELD_GROUP = {
    'epnet_scene': ['301'], # 走下路进入 EPNet 做全局缩放
    'user_profile': ['101', '121', '122', '124', '125', '126', '127', '128', '129'], # 走上路提供给 Gate NN 与 PPNet
    'user_behavior': ['109_14', '110_14', '127_14', '150_14'],                       # 同上，走上路的个性化门控
    'item_and_cross': ['205', '206', '207', '210', '216', '508', '509', '702', '853']  # 走底座并被 "click 塔" 吸收
}

def parse_features_line(feat_str: str):
    """把极其松散的 feature_list 解析为 {field_id: [(feature_id, feature_value)...]} 字典"""
    res = defaultdict(list)
    if pd.isna(feat_str) or not feat_str: return res
    for item in str(feat_str).split(SEP_LIST):
        if not item or SEP_FID not in item: continue
        try:
            fid, rest = item.split(SEP_FID, 1)
            if SEP_IDV in rest:
                feat_id, feat_val = rest.split(SEP_IDV, 1)
            else:
                feat_id, feat_val = rest, '1.0'
            res[fid.strip()].append((feat_id.strip(), float(feat_val)))
        except: pass
    return res

# parse_features_line 函数在干什么？ 假如输入（它收到的原始字符串）是："301\x02456\x031.0\x01101\x02abc\x031.0"
#     split(SEP_LIST) 会把它劈成两个小块："301\x02456\x031.0" 和 "101\x02abc\x031.0"
#     split(SEP_FID) 劈开第一个小块，得到领域 fid = "301", 和内容 "456\x031.0"
#     split(SEP_IDV) 劈开内容，得到特征 feat_id = "456", 值 feat_val = 1.0 得到什么：最终将那堆乱码变成一个规整的字典：{"301": [("456", 1.0)], "101": [("abc", 1.0)]}。

In [ ]:
# ========================================
# 2. 从头扫描一次，构建全局共享词表 (仅依赖 Train 避免泄露)
# ========================================
MIN_FREQ = 5 # 极低频特征直接丢去 OOV (idx=1)
field_vocab_counter = defaultdict(lambda: defaultdict(int))

def count_vocab(filepath, has_labels=True, chunksize=300_000):
    # Skeleton 与 Common 格式有略微不同
    target_col_idx = 5 if has_labels else 2
    for chunk in pd.read_csv(filepath, header=None, chunksize=chunksize, usecols=[target_col_idx], names=['feature_list'], dtype=str):
        for f_str in chunk['feature_list']:
            parsed = parse_features_line(f_str)
            for fid, fvals in parsed.items():
                for f_id, _ in fvals:
                    field_vocab_counter[fid][f_id] += 1

In [ ]:

print("\n== 1. 开始构建全局词表 (Vocab Generator) ==")
print("-> Scanning TRAIN_COMMON...")
count_vocab(TRAIN_COMMON, has_labels=False)
print("-> Scanning TRAIN_SKELETON...")
count_vocab(TRAIN_SKELETON, has_labels=True)


global_vocab = {}
vocab_sizes = {}
for fid, counter in field_vocab_counter.items():
    # 截断低频，并生成密集 Index (0为 padding留用，1为 OOV留用)
    valid_feats = [feat for feat, count in counter.items() if count >= MIN_FREQ]
    feat_to_idx = {feat: idx + 2 for idx, feat in enumerate(valid_feats)}  #enumerate 为它们打上 0, 1, 2... 的标号，然后 +2。意味着最开始的有效ID是 2。
    global_vocab[fid] = feat_to_idx # 保存对照表
    vocab_sizes[fid] = len(feat_to_idx) + 2 
#得到了什么： 得到了字典 global_vocab { "301": { "456": 2, "789": 3 ...}, ... }。 保存了 vocab_sizes.json (记录了每个领域有多少张身份证)。在你未来用 Pytorch 写模型时，你直接读取这个 json，把它传给 nn.Embedding(vocab_size, embedding_dim)，不浪费一丝显存。

with open(OUT_DIR / 'global_vocab.pkl', 'wb') as f:
    pickle.dump(global_vocab, f)
with open(OUT_DIR / 'vocab_sizes.json', 'w') as f:
    json.dump(vocab_sizes, f, indent=2)

print(f"✅ 全局词表构建完成! 共有 {len(global_vocab)} 个维度(Field).")
for f in ['301', '101', '205']:
    if f in vocab_sizes:
        print(f"   - [Field {f}] 词表大小: {vocab_sizes[f]}")


== 1. 开始构建全局词表 (Vocab Generator) ==
-> Scanning TRAIN_COMMON...
-> Scanning TRAIN_SKELETON...
-> Scanning TRAIN_SKELETON...
✅ 全局词表构建完成! 共有 23 个维度(Field).
   - [Field 301] 词表大小: 5
   - [Field 101] 词表大小: 2
   - [Field 205] 词表大小: 912397
✅ 全局词表构建完成! 共有 23 个维度(Field).
   - [Field 301] 词表大小: 5
   - [Field 101] 词表大小: 2
   - [Field 205] 词表大小: 912397


In [5]:
# 如果你是重新打开 notebook，需要确保先加载之前处理好的 global_vocab
if 'global_vocab' not in globals():
    print("⏳ 正在从本地读取 global_vocab.pkl ...")
    with open(OUT_DIR / 'global_vocab.pkl', 'rb') as f:
        global_vocab = pickle.load(f)
    print(f"✅ global_vocab 恢复成功! 共有 {len(global_vocab)} 个维度(Field).")


⏳ 正在从本地读取 global_vocab.pkl ...
✅ global_vocab 恢复成功! 共有 23 个维度(Field).
✅ global_vocab 恢复成功! 共有 23 个维度(Field).


In [6]:
# ========================================
# 3.1 构建 Common SQLite DB
# ========================================
def build_common_sqlite_db(com_path):
    com_db_path = OUT_DIR / f"{com_path.stem}.db"
    if com_db_path.exists():
        print(f"-> 发现已存在的 Common DB: {com_db_path}，跳过构建。")
        return com_db_path

    print(f"-> 正在构建 Common SQLite DB: {com_db_path} (一次性扫描文件，但内存占用低) ...")
    conn = sqlite3.connect(str(com_db_path))
    cur = conn.cursor()
    cur.execute('CREATE TABLE IF NOT EXISTS common (common_idx TEXT PRIMARY KEY, feature_list TEXT)')
    conn.commit()
    # 使用事务批量写入以加速
    insert_sql = 'INSERT OR REPLACE INTO common(common_idx, feature_list) VALUES (?, ?)' 
    batch = []
    batch_size = 20000
    for i, row in enumerate(pd.read_csv(com_path, header=None, usecols=[0, 2], names=['common_idx', 'feature_list'], dtype=str, chunksize=100000)):
        for _, r in row.iterrows():
            batch.append((r['common_idx'], r['feature_list']))
            if len(batch) >= batch_size:
                cur.executemany(insert_sql, batch)
                conn.commit()
                batch = []
        # 小的进度打印
        print(f"    - 已扫描 { (i+1) * 100000 } 行 common 文件...", end='\r')
    if batch:
        cur.executemany(insert_sql, batch)
        conn.commit()
    cur.execute('CREATE INDEX IF NOT EXISTS idx_common_idx ON common(common_idx)')
    conn.commit()
    cur.close()
    conn.close()
    print(f"\n-> Common SQLite DB 构建完成: {com_db_path}")
    return com_db_path

In [7]:
# ========================================
# 3.2 数据集 Join 与 模型级 Tensor 数据装配
# ========================================
def process_dataset_to_model_ready_parquet(skel_path, com_path, out_file, vocab, is_train=True, chunksize=150_000):
    print(f"\n== 处理 {'Train' if is_train else 'Test'} 面向模型的输入集 ==")
    
    com_db_path = OUT_DIR / f"{com_path.stem}.db"
    if not com_db_path.exists():
        raise FileNotFoundError(f"找不到 Common DB: {com_db_path}，请先执行构建。")

    # 打开 DB 连接（读模式）
    conn = sqlite3.connect(str(com_db_path))
    cur = conn.cursor()

    print(f"-> 流式转换 Skeleton 并 Join ({skel_path.name})...")
    
    parquet_writer = None
    total_rows = 0
    chunk_count = 0

    for chunk in pd.read_csv(skel_path, header=None, chunksize=chunksize, 
                             usecols=[0, 1, 2, 3, 5], 
                             names=['sample_id', 'click', 'conversion', 'common_idx', 'feature_list'], 
                             dtype={'click': float, 'conversion': float, 'common_idx': str, 'feature_list': str}):
        # 处理标签，并清洗掉（点击=0，转化=1）这种非法漏斗数据
        chunk['click'] = chunk['click'].fillna(0).astype(int)
        chunk['conversion'] = chunk['conversion'].fillna(0).astype(int)
        if is_train:
            chunk = chunk[~((chunk['click'] == 0) & (chunk['conversion'] == 1))]

        # 提取本 chunk 需要的 common_idx 列表，去重以减少 DB 查询开销
        needed_common = chunk['common_idx'].dropna().unique().tolist()

        com_dict = {}
        if needed_common:
            # 分批查询，避免生成过长的 IN 子句
            batch_size = 1000
            for i in range(0, len(needed_common), batch_size):
                sub = needed_common[i:i+batch_size]
                placeholders = ','.join(['?'] * len(sub))
                sql = f"SELECT common_idx, feature_list FROM common WHERE common_idx IN ({placeholders})"
                cur.execute(sql, sub)
                for k, v in cur.fetchall():
                    com_dict[k] = parse_features_line(v)

        records = []
        for _, row in chunk.iterrows():
            feats_skel = parse_features_line(row['feature_list'])
            feats_com = com_dict.get(row['common_idx'], {})

            # 合并当前 Sample 的两个特征集
            all_feats = defaultdict(list)
            for f_dict in [feats_skel, feats_com]:
                for fid, fvals in f_dict.items():
                    all_feats[fid].extend(fvals)

            record = {
                'sample_id': row['sample_id'],
                'click': row['click'],
                'conversion': row['conversion'],
                'sample_weight_conv': 5.0 if row['conversion'] == 1 else 1.0 
            }

            for group_name, fids in FIELD_GROUP.items():
                group_indices = []
                group_values = []
                for fid in fids:
                    for feat_id, feat_val in all_feats.get(fid, []):
                        idx = vocab.get(fid, {}).get(feat_id, 1) # OOV(1)
                        group_indices.append(idx)
                        group_values.append(feat_val)
                record[f'{group_name}_idx'] = group_indices
                record[f'{group_name}_val'] = group_values

            records.append(record)

        if not records:
            continue

        chunk_df = pd.DataFrame(records)
        table = pa.Table.from_pandas(chunk_df)

        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(out_file, table.schema)

        parquet_writer.write_table(table)

        total_rows += len(chunk_df)
        chunk_count += 1
        print(f"     已落盘 Chunk {chunk_count} ({total_rows} 行)...")

        del chunk_df
        del table
        del records
        gc.collect()

        if chunk_count >= 10:
            print(f"     [!] 达到抽样阈值 (10 Chunks)，提前结束该数据集处理。")
            break

    if parquet_writer is not None:
        parquet_writer.close()

    cur.close()
    conn.close()

    print(f"✅ Saved: {out_file.name}, 有效样本数: {total_rows}")

In [ ]:
# ========================================
# 3.3 执行 - 构建 SQLite DB
# ========================================
print("正在检查并构建 Common SQLite DB...")
#build_common_sqlite_db(TRAIN_COMMON)
build_common_sqlite_db(TEST_COMMON)
print("DB 检查/构建完成！")

正在检查并构建 Common SQLite DB...
-> 正在构建 Common SQLite DB: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/processed/common_features_train.db (一次性扫描文件，但内存占用低) ...
    - 已扫描 800000 行 common 文件...
-> Common SQLite DB 构建完成: /root/autodl-tmp/Sharebottom_PEPNet/datapreprocess/workspace/processed/common_features_train.db
DB 检查/构建完成！


In [9]:
# ========================================
# 3.4 执行 - 处理数据并落盘
# ========================================
#process_dataset_to_model_ready_parquet(TRAIN_SKELETON, TRAIN_COMMON, OUT_DIR / 'train_model_input.parquet', global_vocab, is_train=True)
process_dataset_to_model_ready_parquet(TEST_SKELETON, TEST_COMMON, OUT_DIR / 'test_model_input.parquet', global_vocab, is_train=False)

print("\n🎉 数据预处理与张量映射全部完成！")


== 处理 Test 面向模型的输入集 ==
-> 流式转换 Skeleton 并 Join (sample_skeleton_test.csv)...
     已落盘 Chunk 1 (150000 行)...
     已落盘 Chunk 1 (150000 行)...
     已落盘 Chunk 2 (300000 行)...
     已落盘 Chunk 2 (300000 行)...
     已落盘 Chunk 3 (450000 行)...
     已落盘 Chunk 3 (450000 行)...
     已落盘 Chunk 4 (600000 行)...
     已落盘 Chunk 4 (600000 行)...
     已落盘 Chunk 5 (750000 行)...
     已落盘 Chunk 5 (750000 行)...
     已落盘 Chunk 6 (900000 行)...
     已落盘 Chunk 6 (900000 行)...
     已落盘 Chunk 7 (1050000 行)...
     已落盘 Chunk 7 (1050000 行)...
     已落盘 Chunk 8 (1200000 行)...
     已落盘 Chunk 8 (1200000 行)...
     已落盘 Chunk 9 (1350000 行)...
     已落盘 Chunk 9 (1350000 行)...
     已落盘 Chunk 10 (1500000 行)...
     [!] 达到抽样阈值 (10 Chunks)，提前结束该数据集处理。
✅ Saved: test_model_input.parquet, 有效样本数: 1500000

🎉 数据预处理与张量映射全部完成！
     已落盘 Chunk 10 (1500000 行)...
     [!] 达到抽样阈值 (10 Chunks)，提前结束该数据集处理。
✅ Saved: test_model_input.parquet, 有效样本数: 1500000

🎉 数据预处理与张量映射全部完成！


In [10]:
# ========================================
# 4. 数据质量检视 (测试直接喂给 DataLoader 的长相)
# ========================================
print("== 加载组装好的 Train Parquet 预览 ==")
final_train_df = pd.read_parquet(OUT_DIR / 'train_model_input.parquet')
display(final_train_df.head())

print("\n[Click/CTR 均值]:", final_train_df['click'].mean())
print("[Conversion/CVR 均值]:", final_train_df['conversion'].mean())

print("\nEPNet 输入特征数组示例 (第一条数据):")
print("EPNet index:", final_train_df.iloc[0]['epnet_scene_idx'])

print("\nGate NN 用户侧特征数组示例 (第一条数据):")
print("Profile index:", final_train_df.iloc[0]['user_profile_idx'])
print("Behavior index:", final_train_df.iloc[0]['user_behavior_idx'])

== 加载组装好的 Train Parquet 预览 ==


,sample_id,click,conversion,sample_weight_conv,epnet_scene_idx,epnet_scene_val,user_profile_idx,user_profile_val,user_behavior_idx,user_behavior_val,item_and_cross_idx,item_and_cross_val
0,1,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[117, 1290, 116, 10, 11, 244, 877, 124, 3193, ...","[0.69315, 1.79176, 4.40672, 3.29584, 5.14166, ...","[2, 2, 2, 2, 3, 4, 2, 2]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.69315]"
1,2,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[117, 1290, 116, 10, 11, 244, 877, 124, 3193, ...","[0.69315, 1.79176, 4.40672, 3.29584, 5.14166, ...","[3, 3, 3, 5, 6, 7, 8, 9, 3]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
2,3,1,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[117, 1290, 116, 10, 11, 244, 877, 124, 3193, ...","[0.69315, 1.79176, 4.40672, 3.29584, 5.14166, ...","[4, 4, 4, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
3,4,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[117, 1290, 116, 10, 11, 244, 877, 124, 3193, ...","[0.69315, 1.79176, 4.40672, 3.29584, 5.14166, ...","[5, 5, 5, 21, 22, 23, 24, 5, 4, 3, 4, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.397..."
4,5,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[117, 1290, 116, 10, 11, 244, 877, 124, 3193, ...","[0.69315, 1.79176, 4.40672, 3.29584, 5.14166, ...","[1, 6, 6, 25, 26, 27, 6, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.29584]"



[Click/CTR 均值]: 0.046610435030726954
[Conversion/CVR 均值]: 0.00028600266935824737

EPNet 输入特征数组示例 (第一条数据):
EPNet index: [2]

Gate NN 用户侧特征数组示例 (第一条数据):
Profile index: [1 4 2 2 2 2 2 5]
Behavior index: [   117   1290    116     10     11    244    877    124   3193    126
    127    422    201    852   2292     33   4284     35   1110   1056
     96    348   2310      7    167     12   4770   2171     90    256
   1388    646    645     24    164   2050   1676   3196    489    644
    643   4498     31     32    282    785   1052     78   1228   2316
    159    488     76    157    860    156   1984    538   1418      2
     71    743     70    188    153      9   1586     14    381     16
     19   1020   4942     23   2126   1374   2943    308    705     25
     64     30    146     62    599     61    754   1132    466     56
    354   1915   1822     52     51      5    136      6    459   2945
    454   2160   1752     47    670     17     20    134     29    956
   1081   2754   2

In [11]:
# ========================================
# 4. 数据质量检视 (测试直接喂给 DataLoader 的长相)
# ========================================
print("== 加载组装好的 Test Parquet 预览 ==")
final_train_df = pd.read_parquet(OUT_DIR / 'test_model_input.parquet')
display(final_train_df.head())

print("\n[Click/CTR 均值]:", final_train_df['click'].mean())
print("[Conversion/CVR 均值]:", final_train_df['conversion'].mean())

print("\nEPNet 输入特征数组示例 (第一条数据):")
print("EPNet index:", final_train_df.iloc[0]['epnet_scene_idx'])

print("\nGate NN 用户侧特征数组示例 (第一条数据):")
print("Profile index:", final_train_df.iloc[0]['user_profile_idx'])
print("Behavior index:", final_train_df.iloc[0]['user_behavior_idx'])

== 加载组装好的 Test Parquet 预览 ==


,sample_id,click,conversion,sample_weight_conv,epnet_scene_idx,epnet_scene_val,user_profile_idx,user_profile_val,user_behavior_idx,user_behavior_val,item_and_cross_idx,item_and_cross_val
0,1,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[1720, 117, 1290, 1746, 116, 10, 11, 731, 244,...","[1.09861, 0.69315, 1.79176, 1.09861, 4.51086, ...","[102, 28, 75, 430, 46622, 429, 426, 424, 427, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,2,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[1720, 117, 1290, 1746, 116, 10, 11, 731, 244,...","[1.09861, 0.69315, 1.79176, 1.09861, 4.51086, ...","[43, 25, 53, 200, 228, 199, 2870, 198, 24176, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
2,3,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[1720, 117, 1290, 1746, 116, 10, 11, 731, 244,...","[1.09861, 0.69315, 1.79176, 1.09861, 4.51086, ...","[23239, 13, 13289, 10893, 16082, 26891, 403, 1...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
3,4,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[1720, 117, 1290, 1746, 116, 10, 11, 731, 244,...","[1.09861, 0.69315, 1.79176, 1.09861, 4.51086, ...","[1, 9, 75, 273, 48, 47, 1, 7, 19, 34]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 4.02535, 2..."
4,5,0,0,1.0,[2],[1.0],"[1, 4, 2, 2, 2, 2, 2, 5]","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[1720, 117, 1290, 1746, 116, 10, 11, 731, 244,...","[1.09861, 0.69315, 1.79176, 1.09861, 4.51086, ...","[70759, 18, 56347, 132, 151, 10048, 133, 15214...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."



[Click/CTR 均值]: 0.04698733333333333
[Conversion/CVR 均值]: 0.00034933333333333333

EPNet 输入特征数组示例 (第一条数据):
EPNet index: [2]

Gate NN 用户侧特征数组示例 (第一条数据):
Profile index: [1 4 2 2 2 2 2 5]
Behavior index: [   1720     117    1290    1746     116      10      11     731     244
     877     802     516     182     124    3193     126     127    1435
     422     201     177     202     176    2576    1531     852    2292
      33    4284     624      35      36    1110     118    1056      96
     348     524    2310      93       7     167      12    4770    2171
      90      88     256    1388    1209     565     646     645      24
     164    2050    1676     540    3196     489     644     643    4498
      31      32     555     161     785     282    1052      78    1228
    2316     159     488     487     377      76     157     714     860
     125     156     154    1984     538    1418       2      71     743
      70     188     153     770       9     712      14     381      